In [2]:
# Calculate slopes for RD, then measure DTI values and rpbm from maps using ROI and negative slope areas only, save as csv file
# Gabrielle baxter 08/07/2026

import os
import numpy as np
import nibabel as nib
from nibabel.processing import conform 
from dipy.io.image import load_nifti,save_nifti
from scipy.stats import spearmanr
import pandas as pd

# CHANGED FOR WRAPPER: run_heal_pipeline.sh sets the HEAL_PARENT environment
# variable, otherwise the hard-coded default below is used (no trailing /)
parent_folder = os.environ.get('HEAL_PARENT', '/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075').rstrip('/')
print('Parent folder:', parent_folder)

# Run slopes
diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']
diff_times = np.array([float(s[:-2])*0.001 for s in diffusion_times])

dti_params = ['RD']
map_directory = os.path.join(parent_folder,'maps')
if os.path.exists(map_directory): 
    available_maps = os.listdir(map_directory)
    output_directory = os.path.join(parent_folder,'slopes')
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    for param in dti_params:
        print(param)
        # Read first map
        dt = diffusion_times[0]
        relevant_maps = [s for s in available_maps if dt in s]
        matching_map = [s for s in relevant_maps if param in s]
        matching_map = [s for s in matching_map if not 'colFA' in s]
        matching_map.sort()
        current_map_file = os.path.join(map_directory,matching_map[-1])
        print(current_map_file)
        current_map, map_affine, map_img = load_nifti(current_map_file, return_img=True)
        nx,ny,nz = np.shape(current_map)
        # Set up array
        maps = np.zeros([nx,ny,nz,len(diffusion_times)])
        maps[:,:,:,0] = current_map

        for i in range(1,len(diffusion_times)):
            dt = diffusion_times[i]
            relevant_maps = [s for s in available_maps if dt in s]
            matching_map = [s for s in relevant_maps if param in s]
            matching_map = [s for s in matching_map if not 'colFA' in s]
            matching_map.sort()
            if diffusion_times[i] == '0022ms':
                current_map_file = os.path.join(map_directory,matching_map[-1])
            else:
                matching_map = matching_map[0]
                current_map_file = os.path.join(map_directory,matching_map)
            print(current_map_file)
            current_map, map_affine, map_img = load_nifti(current_map_file, return_img=True)
            maps[:,:,:,i] = current_map

        # Pre assign arrays
        slopes = np.zeros([nx,ny,nz])
        r2_vals = np.zeros([nx,ny,nz])
        rho_map = np.zeros([nx,ny,nz])
    
        # Iterate through voxels
        for i in range(nx):
            for j in range(ny):
                for k in range(nz):
                    y = maps[i, j, k, :]
                    if np.sum(y) > 0: # don't bother if all 0
                    
                        # Fit line y = m*x + b
                        m, b = np.polyfit(diff_times, y, 1)
                        slopes[i, j, k] = m
                        
                        # Compute R^2
                        y_pred = m * diff_times + b
                        ss_res = np.sum((y - y_pred) ** 2)
                        ss_tot = np.sum((y - np.mean(y)) ** 2)
                        r2 = 1 - ss_res/ss_tot if ss_tot > 0 else 0
                        r2_vals[i, j, k] = r2

                        #
                        rho, _ = spearmanr(diff_times, y)
                        rho_map[i, j, k] = rho

        slopes = slopes.astype(np.float32)
        save_filename = os.path.join(output_directory,param + 'slope.nii.gz')
        save_nifti(save_filename,slopes,map_affine)

        r2_vals = r2_vals.astype(np.float32)
        save_filename = os.path.join(output_directory,param + 'r2.nii.gz')
        save_nifti(save_filename,r2_vals,map_affine)

        rho_map = rho_map.astype(np.float32)
        save_filename = os.path.join(output_directory,param + 'rho.nii.gz')
        save_nifti(save_filename,rho_map,map_affine)

# Read DTI parameters from maps - cornell - negative slope ROIs, masked registration, exclude outliers optional
import os
import numpy as np
from dipy.io.image import load_nifti
import pandas as pd

exclude_outliers = 'y'
output_csv = os.path.join(parent_folder,'dti_rpbm_results.csv')

rows = {}

roi_names = ["right_masseter","left_masseter","right_temporalis","left_temporalis"]
diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']
map_names = ['RD','AD','FA']

subject = os.path.basename(parent_folder)
roi_file = os.path.join(parent_folder, f"{subject}_reformatdwi.nii.gz")
full_roi, _ = load_nifti(roi_file)

# use slope to edit ROI
slope,_ = load_nifti(os.path.join(parent_folder,'slopes', 'RDslope.nii.gz'))
roi_array = full_roi.copy()
roi_array[slope > 0] = 0

subject = folder_name = os.path.basename(parent_folder)
maps_folder = os.path.join(parent_folder,'maps')
matched_files = [s for s in os.listdir(maps_folder) if '.nii.gz' in s]
print(matched_files)
for i, roi_name in enumerate(roi_names, start=1):

    mask = (roi_array == i)
    n_voxels = int(np.sum(mask))
    roi_volume = n_voxels * 8

    key = (subject, roi_name)
    rows[key] = {
        "Subject": subject,
        "ROI": roi_name,
        "n_voxels": int(np.sum(roi_array == i)),
        "roi_volume_mm3": int(np.sum(roi_array == i)) * 8
    }

    # ------------------ RPBM MAPS ------------------

    rpbm_folder = os.path.join(parent_folder, "rpbm_mapping")
    rpbm_maps = ["a", "kappa"]

    for map_name in rpbm_maps:

        map_path = os.path.join(rpbm_folder, f"{map_name}.nii")
        print(map_path)

        current_map, _ = load_nifti(map_path)
        values = current_map[mask]
        values = values[np.isfinite(values)]

        if values.size == 0:
            mean_val = np.nan
            std_val = np.nan
            med_val = np.nan
        else:
            if exclude_outliers == 'y':
                lower, upper = np.percentile(values, [5, 95])
                values = values[(values >= lower) & (values <= upper)]

            if values.size == 0:
                mean_val = np.nan
                std_val = np.nan
                med_val = np.nan
            else:
                mean_val = np.mean(values)
                std_val = np.std(values)
                med_val = np.median(values)

        rows[key][f"{map_name}_mean"] = mean_val
        rows[key][f"{map_name}_std"] = std_val
        rows[key][f"{map_name}_median"] = med_val

    # ------------------ DTI MAPS ------------------

    for dt in diffusion_times:
        for map_name in map_names:

            map_files = [s for s in matched_files if dt in s and map_name in s]
            if len(map_files) > 0:
                map_files.sort()
                roi_map_file = [s for s in map_files if roi_name in s]
                if len(roi_map_file) > 0:
                    map_file = roi_map_file[0]
                else:
                    map_file = map_files[-1]

                map_path = os.path.join(maps_folder, map_file)
                print(map_path)
                current_map, _ = load_nifti(map_path)

                values = current_map[mask]

                if values.size == 0:
                    mean_val = np.nan
                    std_val = np.nan
                    med_val = np.nan
                else:
                    values = values[np.isfinite(values)]

                    if values.size == 0:
                        mean_val = np.nan
                        std_val = np.nan
                        med_val = np.nan
                    else:
                        if exclude_outliers == 'y':
                            lower, upper = np.percentile(values, [5, 95])
                            values_trimmed = values[
                                (values >= lower) &
                                (values <= upper)
                            ]
                            mean_val = np.mean(values_trimmed)
                            std_val = np.std(values_trimmed)
                            med_val = np.median(values_trimmed)
                        else:
                            mean_val = np.mean(values)
                            std_val = np.std(values)
                            med_val = np.median(values)

                rows[key][f"{map_name}_mean_{dt}"] = mean_val
                rows[key][f"{map_name}_std_{dt}"] = std_val
                rows[key][f"{map_name}_median_{dt}"] = med_val

df = pd.DataFrame(list(rows.values()))
df = df.dropna()
df.to_csv(output_csv,index=False) # CHANGED FOR WRAPPER: output_csv is already a full path

RD
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0022ms_RD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0042ms_RD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0081ms_RD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0156ms_RD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2075/maps/0300ms_RD.nii.gz


/var/folders/rl/51t1ybms78ggvyh_9h7rm87h0000gr/T/ipykernel_52615/3403694030.py:81: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(diff_times, y)


['0156ms_FA_withinscanreg.nii.gz', '0081ms_FA_withinscanreg.nii.gz', '0300ms_L3_withinscanreg.nii.gz', '0156ms_RD.nii.gz', '0156ms_L2.nii.gz', '0042ms_L3_withinscanreg.nii.gz', '0300ms_RD_withinscanreg.nii.gz', '0300ms_L2.nii.gz', '0300ms_RD.nii.gz', '0042ms_AD_withinscanreg.nii.gz', '0300ms_L2_withinscanreg.nii.gz', '0081ms_L2.nii.gz', '0081ms_RD.nii.gz', '0022ms_RD.nii.gz', '0022ms_L2.nii.gz', '0042ms_RD_withinscanreg.nii.gz', '0300ms_AD_withinscanreg.nii.gz', '0042ms_L2_withinscanreg.nii.gz', '0042ms_RD.nii.gz', '0042ms_L2.nii.gz', '0300ms_FA_withinscanreg.nii.gz', '0081ms_L3_withinscanreg.nii.gz', '0300ms_FA.nii.gz', '0156ms_L3_withinscanreg.nii.gz', '0081ms_FA.nii.gz', '0022ms_FA.nii.gz', '0156ms_AD.nii.gz', '0042ms_FA_withinscanreg.nii.gz', '0156ms_L3.nii.gz', '0042ms_FA.nii.gz', '0081ms_L3.nii.gz', '0022ms_L3.nii.gz', '0081ms_L2_withinscanreg.nii.gz', '0156ms_RD_withinscanreg.nii.gz', '0042ms_AD.nii.gz', '0156ms_L2_withinscanreg.nii.gz', '0081ms_RD_withinscanreg.nii.gz', '0300ms